# Walk-forward demo

Run SMA cross across rolling train/test windows.

In [ ]:
from datetime import date
from blackorca.data.catalog import Catalog
from blackorca.research.walk_forward import run_walk_forward, WalkForwardWindow
from blackorca.backtest.runner import run_backtest
from blackorca.risk.limits import RiskLimits
from blackorca.strategies.examples.sma_cross import SmaCross

cat = Catalog()
bars = cat.read_bars('NVDA', date(2020,1,1), date.today())
ts = bars['as_of'].to_list()
relaxed = RiskLimits(per_order_max_notional=1e9, max_position_pct=0.5)

def evaluate(w: WalkForwardWindow):
    s = SmaCross(symbol='NVDA', fast=10, slow=30, target_weight=0.2)
    r = run_backtest(s, symbols=['NVDA'], start=w.test_start, end=w.test_end, capital=1_000_000, catalog=cat, risk_limits=relaxed)
    return {'sharpe': r.metrics.get('sharpe',0), 'total_return': r.metrics.get('total_return',0)}

result = run_walk_forward(ts, train_days=252, test_days=63, embargo_days=5, evaluator=evaluate)
print(result.summary)
result.to_polars()
